# 📖 Okun Bible Chapter Cleaner — Colab Edition
**Processes page `.txt` files (L + R columns) chapter by chapter using Claude API**

---
### Quick start
1. Get an API key at [console.anthropic.com](https://console.anthropic.com) — new accounts get free credits
2. Store the key as a **Colab Secret** named `ANTHROPIC_API_KEY` (🔑 icon in left sidebar)
3. Upload your page files to Google Drive
4. Set your paths in **Cell 3**, then **Run All**

### File naming convention expected
```
okun_bible_part1_p001L.txt   ← left column, page 1
okun_bible_part1_p001R.txt   ← right column, page 1
okun_bible_part1_p001.png    ← scan image (optional but improves accuracy)
```

In [ ]:
# ── Cell 1: Install & imports ────────────────────────────────────────────────
!pip install anthropic --quiet

import anthropic, base64, json, os, re, time
from pathlib import Path
from google.colab import drive, userdata

print('✔ Ready')

✔ Ready


In [ ]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────────────
drive.mount('/content/drive')
print('✔ Drive mounted')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✔ Drive mounted


In [ ]:
# ── Cell 3: CONFIGURE HERE ───────────────────────────────────────────────────

# API key — stored as Colab Secret or pasted directly
try:
    API_KEY = userdata.get('OKUNCLEAN')
    print('✔ API key loaded from Colab Secrets')

# Folder containing your L/R .txt files and optional .png scans
PAGES_FOLDER  = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/images'   # ← change this
OUTPUT_FOLDER = '/content/drive/MyDrive/Colab Notebooks/Okun digitization/output'  # ← change this

# Books to process — Okun display name : standard English name
# Job/Jobu intentionally excluded
BOOKS_TO_PROCESS = {
    'Erin Defidi':   'Psalms',
    'Iwe Oghe':      'Proverbs',
    'Oliwaasu':      'Ecclesiastes',
    'Erin Solomoni': 'Song of Songs',
    'Aisaya':        'Isaiah',
    'Jobu':          'Job'
}

# Model: haiku = cheapest (~$0.001/page), sonnet = most accurate
MODEL       = 'claude-haiku-4-5-20251001'
API_DELAY   = 1.5   # seconds between calls to avoid rate limits

print(f'Pages  : {PAGES_FOLDER}')
print(f'Output : {OUTPUT_FOLDER}')
print(f'Model  : {MODEL}')
print(f'Books  : {list(BOOKS_TO_PROCESS.keys())}')

✔ API key loaded from Colab Secrets
Pages  : /content/drive/MyDrive/Colab Notebooks/Okun digitization/images
Output : /content/drive/MyDrive/Colab Notebooks/Okun digitization/output
Model  : claude-haiku-4-5-20251001
Books  : ['Erin Defidi', 'Iwe Oghe', 'Oliwaasu', 'Erin Solomoni', 'Aisaya', 'Jobu']


In [ ]:
# ── Cell 4: Verse count table ─────────────────────────────────────────────────
VERSE_COUNTS = {
    'Psalms': [
        6,12,8,8,12,10,17,9,20,18,7,8,6,7,5,11,15,50,14,9,13,31,6,10,22,
        12,14,9,11,12,24,11,22,22,28,12,40,22,13,17,13,11,5,26,17,11,9,14,
        20,23,19,9,6,7,23,13,11,11,17,12,8,12,11,10,13,20,7,35,36,5,24,20,
        28,23,10,12,20,72,13,19,16,8,18,12,13,17,7,18,52,17,16,15,5,23,11,
        13,12,9,9,5,8,28,22,35,45,48,43,13,31,7,10,10,9,8,18,19,2,29,176,
        7,8,9,4,8,5,6,5,6,8,8,3,18,3,3,21,26,9,8,24,13,10,7,12,15,21,10,20,14,9,6],
    'Proverbs':     [33,22,35,27,23,35,27,36,18,32,31,28,25,35,33,33,28,24,29,30,31,29,35,34,28,28,27,28,27,33,31],
    'Ecclesiastes': [18,26,22,16,20,12,29,17,18,20,10,14],
    'Song of Songs':[17,17,11,16,16,13,13,14],
    'Isaiah':       [31,22,26,6,30,13,25,22,21,34,16,6,22,32,9,14,14,7,25,6,17,25,18,23,
                     12,21,13,29,24,33,9,20,24,17,10,22,38,22,8,31,29,25,28,28,25,13,15,
                     22,26,11,23,15,12,17,13,12,21,14,21,22,11,12,19,12,25,24],
    'Jeremiah':     [19,37,25,31,31,30,34,22,26,25,23,17,27,22,21,21,27,23,15,18,14,30,
                     40,10,38,24,22,17,32,24,40,44,26,22,19,32,21,28,18,16,18,22,13,30,
                     5,28,7,47,39,46,64,34],
    'Ezekiel':      [28,10,27,17,17,14,27,18,11,22,25,28,23,23,8,63,24,32,14,49,32,31,
                     49,27,17,21,36,26,21,26,18,32,33,31,15,38,28,23,29,49,26,20,27,31,25,24,23,35],
    'Daniel':       [21,49,30,37,31,28,28,27,27,21,45,13],
    'Hosea':        [11,23,5,19,15,11,16,14,17,15,12,14,16,9],
    'Amos':         [15,16,15,13,27,14,17,14,15],
    'Matthew':      [25,23,17,25,48,34,29,34,38,42,30,50,58,36,39,28,27,35,30,34,46,46,39,51,46,75,66,20],
    'Mark':         [45,28,35,41,43,56,37,38,50,52,33,44,37,72,47,20],
    'Luke':         [80,52,38,44,39,49,50,56,62,42,54,59,35,35,32,31,37,43,48,47,38,71,56,53],
    'John':         [51,25,36,54,47,71,53,59,41,42,57,50,38,31,27,33,26,40,42,31,25],
    'Revelation':   [20,29,22,11,14,17,17,13,21,11,19,17,18,20,8,21,18,24,21,15,27,21],
}

def get_verse_count(std_book, chapter):
    counts = VERSE_COUNTS.get(std_book, [])
    if not counts or chapter < 1 or chapter > len(counts): return None
    return counts[chapter - 1]

print(f'✔ Verse table loaded for {len(VERSE_COUNTS)} books')
print(f'   Psalms 119 = {get_verse_count("Psalms", 119)} verses (expect 176)')
print(f'   Psalms 150 = {get_verse_count("Psalms", 150)} verses (expect 6)')

✔ Verse table loaded for 15 books
   Psalms 119 = 176 verses (expect 176)
   Psalms 150 = 6 verses (expect 6)


In [ ]:
# ── Cell 5: File discovery ────────────────────────────────────────────────────

def discover_pages(folder):
    folder = Path(folder)
    if not folder.exists():
        print(f'⚠ Not found: {folder}'); return []
    all_files = list(folder.iterdir())

    # Identify all potential L files, handling "Copy of " prefix
    raw_l_files = []
    for f in all_files:
        if f.suffix == '.txt':
            stem_normalized = f.stem.replace('Copy of ', '').strip()
            if stem_normalized.upper().endswith('L'):
                raw_l_files.append(f)

    l_files = sorted(raw_l_files) # Sort to ensure consistent processing order

    pages = []
    for lf in l_files:
        # Extract the base name (e.g., 'okun_bible_part1_p001')
        stem_without_copy_prefix = lf.stem.replace('Copy of ', '').strip()
        base = stem_without_copy_prefix[:-1] # Remove the 'L'

        # Construct the expected R file path, accounting for "Copy of " prefix
        r_stem_expected = ('Copy of ' + base + 'R') if 'Copy of ' in lf.stem else (base + 'R')
        rf = lf.parent / (r_stem_expected + '.txt')

        # Extract page number for image matching
        match = re.search(r'p(\d+)', base) # Find '001' from 'okun_bible_part1_p001'
        page_num_str = match.group(1) if match else ''

        img = None
        if page_num_str:
            # Search for image files. Prioritize files that contain 'base' + 'L' or 'base' + 'R'
            # or just 'base' in their stem. This is more specific than just page_num_str.
            for f in all_files:
                if f.suffix.lower() in ['.png','.jpg','.jpeg','.webp']:
                    img_stem = f.stem.lower()

                    # Try matching the full left/right stem part of the image
                    if (base + 'L').lower() in img_stem or (base + 'R').lower() in img_stem:
                        img = f
                        break
                    # Fallback: just match the base page ID (e.g. 'okun_bible_part1_p001')
                    elif base.lower() in img_stem:
                        img = f
                        break

        pages.append({
            'page_id':  base, # e.g., 'okun_bible_part1_p001'
            'page_num': int(page_num_str) if page_num_str else 0,
            'L_path':   lf,
            'R_path':   rf  if rf.exists() else None, # Check if the R file actually exists
            'img_path': img,
        })
    print(f'✔ Found {len(pages)} page pairs in {folder}')
    for p in pages[:5]:
        print(f'   p{p["page_num"]:04d}  ' # Changed f-string due to escape char in previous example
              f'L:{"✔" if p["L_path"] else "✗"}  ' # Changed f-string due to escape char in previous example
              f'R:{"✔" if p["R_path"] else "✗"}  ' # Changed f-string due to escape char in previous example
              f'img:{p["img_path"].name if p["img_path"] else "none"}')
    if len(pages) > 5: print(f'   ...and {len(pages)-5} more')
    return pages

def read_txt(path):
    if not path or not Path(path).exists(): return ''
    return Path(path).read_text(encoding='utf-8', errors='replace').strip()

def read_image_b64(path):
    if not path or not Path(path).exists(): return None, None
    data = Path(path).read_bytes()
    ext  = Path(path).suffix.lower().lstrip('.')
    mime = {'png':'image/png','jpg':'image/jpeg','jpeg':'image/jpeg','webp':'image/webp'}.get(ext,'image/jpeg')
    return base64.b64encode(data).decode(), mime

pages = discover_pages(PAGES_FOLDER)

✔ Found 178 page pairs in /content/drive/MyDrive/Colab Notebooks/Okun digitization/images
   p0001  L:✔  R:✔  img:okun_bible_part1_p001R_ill01.png
   p0002  L:✔  R:✔  img:okun_bible_part1_p002R_ill01.png
   p0003  L:✔  R:✔  img:okun_bible_part1_p003R_ill01.png
   p0004  L:✔  R:✔  img:okun_bible_part1_p004R_ill01.png
   p0005  L:✔  R:✔  img:okun_bible_part1_p005R_ill01.png
   ...and 173 more


In [ ]:
# ── Cell 6: Claude helpers ────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=API_KEY)

def call_claude_page(book_display, standard_book, chapter, total_verses,
                     first_verse, last_verse, page_num,
                     left_text='', right_text='', img_b64=None, img_mime=None):
    """Send L+R text + optional image to Claude. Returns list of verse dicts."""
    ocr_block = ''
    if left_text:  ocr_block += f'--- LEFT COLUMN ---\n{left_text}\n'
    if right_text: ocr_block += f'--- RIGHT COLUMN ---\n{right_text}\n'

    prompt = (
        f'You are a precise scripture editor for an Okun-language Bible translation.\n'
        f'Book: {book_display} (= {standard_book}) | Chapter: {chapter} | Page: {page_num}\n'
        f'Verses on this page: {first_verse}–{last_verse} | Total in chapter: {total_verses}\n'
        + (f'Page scan attached — use as primary source.\n' if img_b64 else '')
        + (f'OCR text (secondary, may have errors):\n{ocr_block}\n' if ocr_block else '')
        + 'Instructions:\n'
        '1. Extract each verse. Use image as primary, OCR as secondary.\n'
        '2. Clean OCR noise but DO NOT change any Okun words.\n'
        '3. Preserve ALL diacritical marks (ọ, ẹ, ị, ghọn, etc.) exactly.\n'
        '4. L column = left half, R = right half — a verse may span both.\n'
        '5. Partially visible verse: include what is there + note [continues from/on next page].\n'
        '6. Fully absent verse: [MISSING — not on this page]\n'
        '7. Output ONLY a valid JSON array — no markdown, no extra text.\n'
        'Format: [{"verse":1,"text":"...","status":"found"},...] '
        'status = "found" | "missing" | "partial"'
    )

    content = []
    if img_b64:
        content.append({'type':'image','source':{'type':'base64','media_type':img_mime,'data':img_b64}})
    content.append({'type':'text','text':prompt})

    resp = client.messages.create(model=MODEL, max_tokens=4000,
                                  messages=[{'role':'user','content':content}])
    raw = ''.join(b.text for b in resp.content if hasattr(b,'text'))
    return json.loads(raw.replace('```json','').replace('```','').strip())


def auto_detect_info(left_text, right_text):
    """Ask Claude to detect book/chapter/verse range from raw text."""
    combined = f'LEFT:\n{left_text[:1500]}\n\nRIGHT:\n{right_text[:1500]}'
    prompt = (
        'This is raw OCR from a page of an Okun-language Bible. '
        'Identify: book name (as it appears), chapter number, '
        'first verse on page, last verse on page.\n\n'
        f'{combined}\n\n'
        'Output ONLY JSON: {"book":"...","chapter":N,"first_verse":N,"last_verse":N} '
        'Use null for any value you cannot determine.'
    )
    resp = client.messages.create(model=MODEL, max_tokens=150,
                                  messages=[{'role':'user','content':prompt}])
    raw = resp.content[0].text.replace('```json','').replace('```','').strip()
    return json.loads(raw)

print(f'✔ Claude client ready ({MODEL})')

✔ Claude client ready (claude-haiku-4-5-20251001)


In [ ]:
# ── Cell 7: Result store & save helpers ──────────────────────────────────────
results = {}   # {book_ch_key: {verse_num: {verse, text, status}}}
proc_log = []

def store(book, chapter, verses):
    key = f'{book}|{chapter}'
    if key not in results: results[key] = {}
    for v in verses:
        n = v['verse']
        old = results[key].get(n)
        if old is None or old['status'] == 'missing':
            results[key][n] = v

def chapter_text(book_display, standard_book, chapter):
    key   = f'{book_display}|{chapter}'
    data  = results.get(key, {})
    total = get_verse_count(standard_book, chapter) or (max(data) if data else 0)
    lines = [f'{book_display} {chapter}', '']
    found = missing = 0
    for i in range(1, total + 1):
        v = data.get(i)
        if v and v['status'] != 'missing': lines.append(f'{i} {v["text"]}'); found += 1
        else: lines.append(f'{i} [MISSING]'); missing += 1
    return '\n'.join(lines), found, missing

def save_chapter(book_display, standard_book, chapter):
    out = Path(OUTPUT_FOLDER) / book_display.replace(' ','_')
    out.mkdir(parents=True, exist_ok=True)
    txt, f, m = chapter_text(book_display, standard_book, chapter)
    path = out / f'ch_{chapter:03d}.txt'
    path.write_text(txt, encoding='utf-8')
    return path, f, m

def save_full_book(book_display, standard_book):
    total_ch = len(VERSE_COUNTS.get(standard_book, []))
    lines = []
    for ch in range(1, total_ch + 1):
        t, _, _ = chapter_text(book_display, standard_book, ch)
        lines += [t, '']
    out = Path(OUTPUT_FOLDER) / book_display.replace(' ','_')
    out.mkdir(parents=True, exist_ok=True)
    p = out / f'{book_display.replace(" ","_")}_FULL.txt'
    p.write_text('\n'.join(lines), encoding='utf-8')
    return p

print('✔ Store & save helpers ready')

✔ Store & save helpers ready


In [ ]:
def reconcile_missing_verses():
    print('\nStarting missing verse reconciliation...')
    reconciled_count = 0
    reconciliation_log = []

    # Identify chapters with missing verses
    chapters_with_missing = set()
    for key, chapter_data in results.items():
        book_display, chapter_num = key.split('|')
        std_book = BOOKS_TO_PROCESS.get(book_display, book_display)
        total_verses_in_chapter = get_verse_count(std_book, int(chapter_num)) or 0

        for i in range(1, total_verses_in_chapter + 1):
            if i not in chapter_data or chapter_data[i]['status'] == 'missing':
                chapters_with_missing.add(key)
                break

    if not chapters_with_missing:
        print('No missing verses found to reconcile.')
        return 0, []

    print(f'Found {len(chapters_with_missing)} chapters with missing verses.')

    for key in sorted(list(chapters_with_missing)):
        book_display, chapter_str = key.split('|')
        chapter_num = int(chapter_str)
        std_book = BOOKS_TO_PROCESS.get(book_display, book_display)

        print(f'  Reconciling {book_display} Chapter {chapter_num}...')

        total_verses_in_chapter = get_verse_count(std_book, chapter_num) or 0

        # Find relevant pages
        relevant_page_ids = set()
        for entry in proc_log:
            if entry.get('book') == book_display and entry.get('chapter') == chapter_num:
                relevant_page_ids.add(entry['page'])

        # Include adjacent pages
        page_indices_to_reprocess = set()
        for p_idx, p in enumerate(pages):
            if p['page_id'] in relevant_page_ids:
                page_indices_to_reprocess.add(p_idx)
                if p_idx > 0:
                    page_indices_to_reprocess.add(p_idx - 1)
                if p_idx < len(pages) - 1:
                    page_indices_to_reprocess.add(p_idx + 1)

        pages_to_reprocess = sorted(
            [pages[i] for i in page_indices_to_reprocess],
            key=lambda x: x['page_num']
        )

        print(f'    Re-processing {len(pages_to_reprocess)} pages...')

        current_reconciled_for_chapter = 0

        for page_to_process in pages_to_reprocess:
            page_id = page_to_process['page_id']

            # Avoid duplicate attempts
            if any(
                log_entry.get('page') == page_id and
                log_entry.get('reconciliation_attempt_for') == key
                for log_entry in reconciliation_log
            ):
                continue

            print(f'      Re-evaluating page {page_id}...')

            L = read_txt(page_to_process['L_path'])
            R = read_txt(page_to_process['R_path'])
            if not L and not R:
                continue

            img_b64, img_mime = read_image_b64(page_to_process['img_path'])

            try:
                info = auto_detect_info(L, R)

                global api_calls
                api_calls += 1
                time.sleep(API_DELAY)

                raw_book_page = info.get('book') or ''
                chapter_page = info.get('chapter')
                first_verse_page = info.get('first_verse') or 1
                last_verse_page = info.get('last_verse')

                if not raw_book_page or not chapter_page:
                    continue

                # Match book name
                book_display_page = None
                standard_book_page = None

                for disp, std in BOOKS_TO_PROCESS.items():
                    if (
                        disp.lower() in raw_book_page.lower() or
                        raw_book_page.lower() in disp.lower() or
                        std.lower() in raw_book_page.lower()
                    ):
                        book_display_page = disp
                        standard_book_page = std
                        break

                if not book_display_page:
                    continue

                # 🔒 STRICT CHAPTER MATCH
                if not (book_display_page == book_display and chapter_page == chapter_num):
                    continue

                total_verses_page = get_verse_count(standard_book_page, chapter_page)

                # Count missing BEFORE
                initial_missing = sum(
                    1 for i in range(1, total_verses_in_chapter + 1)
                    if i not in results[key] or results[key][i]['status'] == 'missing'
                )

                # Call Claude
                verses_from_claude = call_claude_page(
                    book_display_page,
                    standard_book_page,
                    chapter_page,
                    total_verses_page,
                    first_verse_page,
                    last_verse_page,
                    page_to_process['page_num'],
                    left_text=L,
                    right_text=R,
                    img_b64=img_b64,
                    img_mime=img_mime
                )

                api_calls += 1
                time.sleep(API_DELAY)

                # FILTER RESULTS SAFELY
                filtered_verses = {}

                for verse_num, verse_data in verses_from_claude.items():
                    if (
                        isinstance(verse_num, int) and
                        1 <= verse_num <= total_verses_in_chapter and
                        first_verse_page <= verse_num <= (last_verse_page or total_verses_in_chapter) and
                        (
                            verse_num not in results[key] or
                            results[key][verse_num]['status'] == 'missing'
                        )
                    ):
                        filtered_verses[verse_num] = verse_data

                # Store only safe verses
                if filtered_verses:
                    store(book_display, chapter_num, filtered_verses)

                # Count missing AFTER
                new_missing = sum(
                    1 for i in range(1, total_verses_in_chapter + 1)
                    if i not in results[key] or results[key][i]['status'] == 'missing'
                )

                found_this_pass = initial_missing - new_missing

                if found_this_pass > 0:
                    current_reconciled_for_chapter += found_this_pass
                    reconciled_count += found_this_pass
                    print(f'        Found {found_this_pass} new verses.')

                reconciliation_log.append({
                    'page': page_id,
                    'book': book_display,
                    'chapter': chapter_num,
                    'status': 'ok',
                    'found': found_this_pass,
                    'reconciliation_attempt_for': key
                })

            except Exception as e:
                print(f'      ⚠ Error on page {page_id}: {e}')
                reconciliation_log.append({
                    'page': page_id,
                    'status': f'error:{e}',
                    'reconciliation_attempt_for': key
                })

        print(f'    Total reconciled for chapter: {current_reconciled_for_chapter}')

    print(f'\nReconciliation complete. Total {reconciled_count} verses fixed.')
    return reconciled_count, reconciliation_log

In [ ]:
# ── Cell 8: MAIN LOOP — processes every page automatically ───────────────────
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

if not pages:
    print('⚠ No pages found — check PAGES_FOLDER in Cell 3')
else:
    api_calls = 0
    skipped   = []

    for idx, page in enumerate(pages):
        print(f'\n[{idx+1}/{len(pages)}] Page {page["page_num"]}  ({page["page_id"]})')

        L = read_txt(page['L_path'])
        R = read_txt(page['R_path'])
        if not L and not R:
            print('  ⚠ Both L and R empty — skip'); skipped.append(page['page_id']); continue

        img_b64, img_mime = read_image_b64(page['img_path'])
        print(f'  L:{len(L)}  R:{len(R)}  img:{page["img_path"].name if page["img_path"] else "none"}')

        # Auto-detect book/chapter/verse range
        try:
            info = auto_detect_info(L, R)
            api_calls += 1; time.sleep(API_DELAY)
        except Exception as e:
            print(f'  ⚠ Auto-detect failed: {e} — skip'); skipped.append(page['page_id']); continue

        raw_book     = info.get('book') or ''
        chapter      = info.get('chapter')
        first_verse  = info.get('first_verse') or 1
        last_verse   = info.get('last_verse')

        if not raw_book or not chapter:
            print('  ⚠ Could not detect book/chapter — skip'); skipped.append(page['page_id']); continue

        # Match to a book in BOOKS_TO_PROCESS
        book_display = standard_book = None
        for disp, std in BOOKS_TO_PROCESS.items():
            if disp.lower() in raw_book.lower() or raw_book.lower() in disp.lower() or std.lower() in raw_book.lower():
                book_display = disp; standard_book = std; break

        if book_display is None:
            print(f'  ⚠ "{raw_book}" not in BOOKS_TO_PROCESS — skip')
            skipped.append(page['page_id']); continue

        total_verses = get_verse_count(standard_book, chapter)
        last_verse   = last_verse or total_verses
        print(f'  → {book_display} {chapter}:{first_verse}–{last_verse}  (of {total_verses})')

        try:
            verses = call_claude_page(
                book_display, standard_book, chapter, total_verses,
                first_verse, last_verse, page['page_num'],
                left_text=L, right_text=R, img_b64=img_b64, img_mime=img_mime
            )
            api_calls += 1; time.sleep(API_DELAY)

            store(book_display, chapter, verses)
            f_cnt = sum(1 for v in verses if v['status'] != 'missing')
            m_cnt = sum(1 for v in verses if v['status'] == 'missing')
            print(f'  ✔ {f_cnt} found, {m_cnt} missing')

            path, cf, cm = save_chapter(book_display, standard_book, chapter)
            print(f'  💾 {path.name}  (chapter total: {cf} found, {cm} still missing)')

            proc_log.append({'page':page['page_id'],'book':book_display,
                             'chapter':chapter,'found':f_cnt,'missing':m_cnt,'status':'ok'})
        except Exception as e:
            print(f'  ⚠ Claude error: {e}')
            proc_log.append({'page':page['page_id'],'status':f'error:{e}'})

    print(f'\n{"="*55}')
    print(f'Done!  Pages:{len(pages)-len(skipped)}  API calls:{api_calls}  Skipped:{len(skipped)}')
    if skipped: print(f'Skipped: {skipped}')

    # --- NEW RECONCILIATION STEP ---
    total_reconciled_verses, recon_log = reconcile_missing_verses()
    proc_log.extend(recon_log) # Add reconciliation attempts to the log
    print(f'\nReconciliation phase: {total_reconciled_verses} verses newly found.')

# ... (rest of Cell 8, if any, for example, a final print statement)


[1/178] Page 1  (okun_bible_part1_p001)
  L:2830  R:2113  img:okun_bible_part1_p001R_ill01.png
  → Jobu 1:10–22  (of None)
  ⚠ Claude error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.content.0.image.source.base64: image exceeds 5 MB maximum: 5297568 bytes > 5242880 bytes'}, 'request_id': 'req_011CZGVuZ29objS868YpUfnm'}

[2/178] Page 2  (okun_bible_part1_p002)
  L:2492  R:2370  img:okun_bible_part1_p002R_ill01.png
  → Jobu 3:1–18  (of None)
  ✔ 22 found, 0 missing
  💾 ch_003.txt  (chapter total: 22 found, 0 still missing)

[3/178] Page 3  (okun_bible_part1_p003)
  L:2292  R:2441  img:okun_bible_part1_p003R_ill01.png
  → Jobu 6:1–17  (of None)
  ⚠ Claude error: Expecting ',' delimiter: line 4 column 32 (char 53)

[4/178] Page 4  (okun_bible_part1_p004)
  L:2435  R:2255  img:okun_bible_part1_p004R_ill01.png
  → Jobu 6:27–None  (of None)
  ✔ 21 found, 0 missing
  💾 ch_006.txt  (chapter total: 21 found, 0 still missing)

[5/178] P

In [ ]:
# ── Cell 9: Save full book files & print summary ──────────────────────────────
for book_display, standard_book in BOOKS_TO_PROCESS.items():
    if not any(k.startswith(f'{book_display}|') for k in results): continue
    full = save_full_book(book_display, standard_book)
    total_ch = len(VERSE_COUNTS.get(standard_book, []))
    done_ch  = sum(1 for k in results if k.startswith(f'{book_display}|'))
    tf = tm = 0
    for ch in range(1, total_ch + 1):
        _, f, m = chapter_text(book_display, standard_book, ch)
        tf += f; tm += m
    print(f'📖 {book_display}  ({done_ch}/{total_ch} chapters)  {tf} found  {tm} [MISSING]')
    print(f'   {full}')

log_path = Path(OUTPUT_FOLDER)/'processing_log.json'
log_path.write_text(json.dumps(proc_log, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'\n📋 Log: {log_path}')

In [ ]:
# ── Cell 10: Preview any chapter ─────────────────────────────────────────────
PREVIEW_BOOK    = 'Erin Defidi'  # ← change
PREVIEW_CHAPTER = 1              # ← change

std = BOOKS_TO_PROCESS.get(PREVIEW_BOOK, PREVIEW_BOOK)
txt, f, m = chapter_text(PREVIEW_BOOK, std, PREVIEW_CHAPTER)
print(txt)
print(f'\n--- {f} found | {m} [MISSING] ---')

In [ ]:
# ── Cell 11: Manually process ONE page ───────────────────────────────────────
# Use to re-run a specific page or test without the full loop

MANUAL_BASE         = 'okun_bible_part1_p003'  # ← page base name (no L/R)
MANUAL_BOOK         = 'Erin Defidi'
MANUAL_CHAPTER      = 1
MANUAL_FIRST_VERSE  = 1
MANUAL_LAST_VERSE   = 6

folder = Path(PAGES_FOLDER)
L  = read_txt(folder / (MANUAL_BASE + 'L.txt'))
R  = read_txt(folder / (MANUAL_BASE + 'R.txt'))
img_b64, img_mime = read_image_b64(next(
    (folder/f'{MANUAL_BASE}{e}' for e in ['.png','.jpg','.jpeg'] if (folder/f'{MANUAL_BASE}{e}').exists()), None))

std   = BOOKS_TO_PROCESS.get(MANUAL_BOOK, MANUAL_BOOK)
total = get_verse_count(std, MANUAL_CHAPTER)
print(f'L:{len(L)} chars  R:{len(R)} chars  img:{"yes" if img_b64 else "no"}  total verses:{total}\n')

verses = call_claude_page(MANUAL_BOOK, std, MANUAL_CHAPTER, total,
                          MANUAL_FIRST_VERSE, MANUAL_LAST_VERSE, MANUAL_BASE,
                          left_text=L, right_text=R, img_b64=img_b64, img_mime=img_mime)

for v in verses:
    tag = '' if v['status']=='found' else f'  [{v["status"].upper()}]'
    print(f'{v["verse"]:3d}  {v["text"]}{tag}')

# Uncomment to save:
# store(MANUAL_BOOK, MANUAL_CHAPTER, verses)
# save_chapter(MANUAL_BOOK, std, MANUAL_CHAPTER)